# NLF shape acceptance test (Colab)

P0 of the shape fix: does NLF's SMPL-X β respond to body shape? Same 8-photo
test and |Δβ| metric as `tools/shape_eval/README.md`.

Why Colab: the NLF multi-person TorchScript hardcodes `cuda:0` in its YOLO
detector, so it needs an NVIDIA GPU. *Runtime → Change runtime type → T4 GPU.*

Upload `shape_eval_colab.zip` (`nlf_eval.py` + `photos/`, built from
`tools/shape_eval/` and `frontend/public/samples/`) into the session's `/content/`.

In [ ]:
# 1. Match the Space: torch 2.9.1 / torchvision 0.24.1 (NLF TorchScript loading is version-sensitive)
import torch, torchvision
print('preinstalled:', torch.__version__, torchvision.__version__)
if not torch.__version__.startswith('2.9.1'):
    !pip -q install torch==2.9.1 torchvision==0.24.1
    print('Installed. Runtime → Restart session, then run this cell again.')

In [ ]:
# 2. GPU, NLF weights (493 MB, per session), unzip inputs
import os
!nvidia-smi --query-gpu=name --format=csv,noheader
NLF = '/content/nlf_l_multi_0.3.2.torchscript'
if not os.path.exists(NLF):
    !wget -q --show-progress -O {NLF} https://github.com/isarandi/nlf/releases/download/v0.3.2/nlf_l_multi_0.3.2.torchscript
!rm -rf /content/se && mkdir -p /content/se && unzip -q -o /content/shape_eval_colab.zip -d /content/se && find /content/se -type f | sort

## `beta_regularizer` sweep

The first two images of the first pass include TorchScript warm-up (~20 s each);
ignore those times.

In [ ]:
# 3. Run
%cd /content/se
!python -u nlf_eval.py --model {NLF} photos/sample-female.jpg photos/sample-male.jpg photos/stress/*.jpg --beta-reg 10 1 0.1 0 2>&1 | grep -v Warning